In [9]:
# ============================================
# 0. 라이브러리 / 경로
# ============================================

import numpy as np
import pandas as pd
from pathlib import Path
from numerai_tools.scoring import numerai_corr

BASE_DIR = Path("/Users/seomichelle/26-1 ESAA:Python/datasets/Numerai")
CACHE_DIR = BASE_DIR / "cache_v52"
DIAGNOSTICS_DIR = BASE_DIR / "diagnostics_v52"

DIAGNOSTICS_DIR.mkdir(parents=True, exist_ok=True)

RIDGE_PATH = BASE_DIR / "ridge_fullrefit_diagnostics.csv"
ELASTIC_PATH = BASE_DIR / "elasticnet_fullrefit_diagnostics.csv"
XGBOOST_PATH = BASE_DIR / "xgboost_fullrefit_diagnostics.csv"

VALIDATION_INFO_PATH = CACHE_DIR / "validation_pca500_lag1_lag2.parquet"

DIAGNOSTICS_CSV = DIAGNOSTICS_DIR / "ridge_elastic_xgboost_v52_diagnostics2.csv"
PER_ERA_CSV = DIAGNOSTICS_DIR / "ridge_elastic_xgboost_v52_per_era_corr.csv"
SUMMARY_CSV = DIAGNOSTICS_DIR / "ridge_elastic_xgboost_v52_local_summary.csv"

print("Ridge:", RIDGE_PATH)
print("ElasticNet:", ELASTIC_PATH)
print("XGBoost:", XGBOOST_PATH)
print("Final:", DIAGNOSTICS_CSV)

Ridge: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/ridge_fullrefit_diagnostics.csv
ElasticNet: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/elasticnet_fullrefit_diagnostics.csv
XGBoost: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/xgboost_fullrefit_diagnostics.csv
Final: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_diagnostics2.csv


In [10]:
# ============================================
# 1. 예측 CSV 및 공식 validation 정보 불러오기
# ============================================

ridge = pd.read_csv(RIDGE_PATH)
elastic = pd.read_csv(ELASTIC_PATH)
xgboost = pd.read_csv(XGBOOST_PATH)

ridge["id"] = ridge["id"].astype(str)
elastic["id"] = elastic["id"].astype(str)
xgboost["id"] = xgboost["id"].astype(str)

validation_info = pd.read_parquet(
    VALIDATION_INFO_PATH,
    columns=["era", "target"],
).reset_index(names="id")

validation_info["id"] = validation_info["id"].astype(str)
validation_info["row_order"] = np.arange(len(validation_info))

print("Ridge shape:", ridge.shape)
print("ElasticNet shape:", elastic.shape)
print("XGBoost shape:", xgboost.shape)
print("Validation shape:", validation_info.shape)

print("Ridge = ElasticNet ID 순서:", ridge["id"].equals(elastic["id"]))
print("Ridge = XGBoost ID 순서:", ridge["id"].equals(xgboost["id"]))

display(ridge.head())
display(elastic.head())
display(xgboost.head())

Ridge shape: (4050241, 2)
ElasticNet shape: (4050241, 2)
XGBoost shape: (4085748, 2)
Validation shape: (4050241, 4)
Ridge = ElasticNet ID 순서: True
Ridge = XGBoost ID 순서: False


,id,prediction
0,n000101811a8a843,0.356760
1,n001e1318d5072ac,0.770565
2,n002a9c5ab785cbb,0.305615
3,n002ccf6d0e8c5ad,0.460300
4,n0041544c345c91d,0.596566


,id,prediction
0,n000101811a8a843,0.386445
1,n001e1318d5072ac,0.784514
2,n002a9c5ab785cbb,0.205114
3,n002ccf6d0e8c5ad,0.532547
4,n0041544c345c91d,0.467096


,id,prediction
0,n000101811a8a843,0.496772
1,n001e1318d5072ac,0.507399
2,n002a9c5ab785cbb,0.498595
3,n002ccf6d0e8c5ad,0.511486
4,n0041544c345c91d,0.510016


In [11]:
# ============================================
# 2. ID 기준 병합
# ============================================

ridge = ridge.rename(
    columns={"prediction": "ridge_prediction"}
)

elastic = elastic.rename(
    columns={"prediction": "elasticnet_prediction"}
)

xgboost = xgboost.rename(
    columns={"prediction": "xgboost_prediction"}
)

prediction_df = validation_info.merge(
    ridge[["id", "ridge_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.merge(
    elastic[["id", "elasticnet_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.merge(
    xgboost[["id", "xgboost_prediction"]],
    on="id",
    how="left",
    validate="one_to_one",
)

prediction_df = prediction_df.sort_values(
    "row_order"
).reset_index(drop=True)

print("병합 결과:", prediction_df.shape)
print("Ridge 결측치:", prediction_df["ridge_prediction"].isna().sum())
print("ElasticNet 결측치:", prediction_df["elasticnet_prediction"].isna().sum())
print("XGBoost 결측치:", prediction_df["xgboost_prediction"].isna().sum())

assert prediction_df["ridge_prediction"].notna().all()
assert prediction_df["elasticnet_prediction"].notna().all()
assert prediction_df["xgboost_prediction"].notna().all()

display(prediction_df.head())

병합 결과: (4050241, 7)
Ridge 결측치: 0
ElasticNet 결측치: 0
XGBoost 결측치: 0


,id,era,target,row_order,ridge_prediction,elasticnet_prediction,xgboost_prediction
0,n000101811a8a843,575,0.5,0,0.356760,0.386445,0.496772
1,n001e1318d5072ac,575,0.5,1,0.770565,0.784514,0.507399
2,n002a9c5ab785cbb,575,0.5,2,0.305615,0.205114,0.498595
3,n002ccf6d0e8c5ad,575,0.0,3,0.460300,0.532547,0.511486
4,n0041544c345c91d,575,0.5,4,0.596566,0.467096,0.510016


In [12]:
# ============================================
# 3. Ridge + ElasticNet + XGBoost rank average
# ============================================

prediction_df["ridge_rank"] = prediction_df.groupby(
    "era"
)["ridge_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["elasticnet_rank"] = prediction_df.groupby(
    "era"
)["elasticnet_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["xgboost_rank"] = prediction_df.groupby(
    "era"
)["xgboost_prediction"].rank(
    method="average",
    pct=True,
).astype(np.float32)

prediction_df["prediction"] = prediction_df[
    [
        "ridge_rank",
        "elasticnet_rank",
        "xgboost_rank",
    ]
].mean(axis=1).astype(np.float32)

diagnostics_df = prediction_df[
    ["id", "prediction"]
].copy()

diagnostics_df.to_csv(
    DIAGNOSTICS_CSV,
    index=False,
)

print("Diagnostics CSV:", DIAGNOSTICS_CSV)
print("shape:", diagnostics_df.shape)
print("prediction min:", diagnostics_df["prediction"].min())
print("prediction max:", diagnostics_df["prediction"].max())

display(diagnostics_df.head())

Diagnostics CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_diagnostics2.csv
shape: (4050241, 2)
prediction min: 0.00013906271
prediction max: 1.0


,id,prediction
0,n000101811a8a843,0.348593
1,n001e1318d5072ac,0.805854
2,n002a9c5ab785cbb,0.306867
3,n002ccf6d0e8c5ad,0.649797
4,n0041544c345c91d,0.665892


In [13]:
# ============================================
# 4. 모델별 예측값 비교
# ============================================

comparison = prediction_df[
    [
        "id",
        "era",
        "ridge_prediction",
        "elasticnet_prediction",
        "xgboost_prediction",
        "ridge_rank",
        "elasticnet_rank",
        "xgboost_rank",
        "prediction",
    ]
].head(10)

display(comparison)

print(
    "최종 CSV = Ridge ID 순서:",
    diagnostics_df["id"].equals(ridge["id"]),
)

print(
    "최종 CSV = ElasticNet ID 순서:",
    diagnostics_df["id"].equals(elastic["id"]),
)

print(
    "최종 CSV = XGBoost ID 순서:",
    diagnostics_df["id"].equals(xgboost["id"]),
)

,id,era,ridge_prediction,elasticnet_prediction,xgboost_prediction,ridge_rank,elasticnet_rank,xgboost_rank,prediction
0,n000101811a8a843,575,0.356760,0.386445,0.496772,0.356760,0.386445,0.302575,0.348593
1,n001e1318d5072ac,575,0.770565,0.784514,0.507399,0.770565,0.784514,0.862482,0.805854
2,n002a9c5ab785cbb,575,0.305615,0.205114,0.498595,0.305615,0.205114,0.409871,0.306867
3,n002ccf6d0e8c5ad,575,0.460300,0.532547,0.511486,0.460300,0.532547,0.956545,0.649797
4,n0041544c345c91d,575,0.596566,0.467096,0.510016,0.596566,0.467096,0.934013,0.665892
5,n0051ab821295c29,575,0.265916,0.205472,0.489830,0.265916,0.205472,0.053827,0.175072
6,n0062826215fe6aa,575,0.423641,0.235694,0.500749,0.423641,0.235694,0.534156,0.397830
7,n008361ac9e9bd47,575,0.054363,0.027718,0.496562,0.054363,0.027718,0.292024,0.124702
8,n009e95486e1d64c,575,0.375894,0.372496,0.500040,0.375894,0.372496,0.494456,0.414282
9,n00b093a02b84295,575,0.409335,0.140916,0.509434,0.409335,0.140916,0.921853,0.490701


최종 CSV = Ridge ID 순서: True
최종 CSV = ElasticNet ID 순서: True
최종 CSV = XGBoost ID 순서: False


In [14]:
# ============================================
# 5. 로컬 Numerai CORR / Sharpe
# ============================================

per_era_rows = []

for era, group in prediction_df.loc[
    prediction_df["target"].notna()
].groupby("era", sort=True):

    score = numerai_corr(
        group[["prediction"]],
        group["target"],
    )

    score_value = float(
        np.asarray(score).reshape(-1)[0]
    )

    per_era_rows.append({
        "era": int(era),
        "corr": score_value,
        "rows": len(group),
    })

per_era_df = pd.DataFrame(per_era_rows)
per_era_df.to_csv(PER_ERA_CSV, index=False)

corr_mean = per_era_df["corr"].mean()
corr_std = per_era_df["corr"].std(ddof=0)
corr_sharpe = corr_mean / corr_std if corr_std > 0 else np.nan

cumulative_corr = per_era_df["corr"].cumsum()

max_drawdown = (
    cumulative_corr.expanding(min_periods=1).max()
    - cumulative_corr
).max()

summary_df = pd.DataFrame({
    "metric": [
        "mean_corr",
        "std_corr",
        "sharpe",
        "max_drawdown",
        "eras",
    ],
    "value": [
        corr_mean,
        corr_std,
        corr_sharpe,
        max_drawdown,
        len(per_era_df),
    ],
})

summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df)

print("Per-era CSV:", PER_ERA_CSV)
print("Summary CSV:", SUMMARY_CSV)

,metric,value
0,mean_corr,0.013309
1,std_corr,0.015436
2,sharpe,0.862188
3,max_drawdown,0.180886
4,eras,649.000000


Per-era CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_per_era_corr.csv
Summary CSV: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_local_summary.csv


In [15]:
# ============================================
# 6. 최종 파일 확인
# ============================================

final_check = pd.read_csv(DIAGNOSTICS_CSV)

print("파일 존재:", DIAGNOSTICS_CSV.exists())
print("저장 경로:", DIAGNOSTICS_CSV)
print("행 수:", len(final_check))
print("컬럼:", final_check.columns.tolist())
print("중복 ID:", final_check["id"].duplicated().sum())
print("결측 prediction:", final_check["prediction"].isna().sum())
print("prediction 최솟값:", final_check["prediction"].min())
print("prediction 최댓값:", final_check["prediction"].max())

display(final_check.head())

파일 존재: True
저장 경로: /Users/seomichelle/26-1 ESAA:Python/datasets/Numerai/diagnostics_v52/ridge_elastic_xgboost_v52_diagnostics2.csv
행 수: 4050241
컬럼: ['id', 'prediction']
중복 ID: 0
결측 prediction: 0
prediction 최솟값: 0.00013906271
prediction 최댓값: 1.0


,id,prediction
0,n000101811a8a843,0.348593
1,n001e1318d5072ac,0.805854
2,n002a9c5ab785cbb,0.306867
3,n002ccf6d0e8c5ad,0.649797
4,n0041544c345c91d,0.665892
